In [1]:
# Estrategia Lay 0x1

import pandas as pd
import numpy as np

# Mostar todas as colunas
pd.set_option('display.max_columns', None)

In [16]:
# Estrategia Lay 0x1 - Get Up Trading

import pandas as pd
import numpy as np

# Mostar todas as colunas
pd.set_option('display.max_columns', None)

data = pd.read_csv("../../data_total/dados_betfair.csv", sep=";")

# Filtar colunas para análise
datatest = data[['Date', 'League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Goals_Min_H', 'Odd_H_Back', 'Odd_A_Back', 'Odd_Over25_FT_Back', 'Odd_BTTS_Yes_Back', 'Odd_CS_0x1_Lay']].copy()

# Concatenar com a base dados_footystats.csv | o nome dos times estão no aruivo team_map.py
dados_footystats = pd.read_csv("../../data_total/dados_footystats.csv", sep=";")

# Escolher as colunas relevantes da base dados_footystats
dados_footystats = dados_footystats[['Date', 'League', 'Home', 'Away', 'Goals_H_HT', 'xG_H']]

FOOTYSTATS_TEAM_MAP = {
    "Roma": "AS Roma",
    "AFC Bournemouth": "Bournemouth",
    "Almería": "Almeria",
    "América Mineiro": "America Mineiro",
    "Angers SCO": "Angers",
    "Athletic Club Bilbao": "Ath Bilbao",
    "Atlético GO": "Atletico GO",
    "Atlético Madrid": "Atl. Madrid",
    "Atlético PR": "Athletico-PR",
    "Bayern München": "Bayern Munich",
    "Boavista FC": "Boavista",
    "Borussia Dortmund": "Dortmund",
    "Borussia M'gladbach": "B. Monchengladbach",
    "Botafogo": "Botafogo RJ",
    "Brighton & Hove Albion": "Brighton",
    "CA Osasuna": "Osasuna",
    "CD Nacional": "Nacional",
    "CD Tondela": "Tondela",
    "Ceará": "Ceara",
    "Celta de Vigo": "Celta Vigo",
    "Chapecoense": "Chapecoense-SC",
    "Criciúma": "Criciuma",
    "Cuiabá": "Cuiaba",
    "Cádiz": "Cadiz CF",
    "Darmstadt 98": "Darmstadt",
    "Deportivo Alavés": "Alaves",
    "Elche CF": "Elche",
    "Estrela Amadora": "Estrela",
    "FC Arouca": "Arouca",
    "FC Barcelona": "Barcelona",
    "FC Vizela": "Vizela",
    "Famalicão": "Famalicao",
    "Flamengo": "Flamengo RJ",
    "GD Chaves": "Chaves",
    "GD Estoril Praia": "Estoril",
    "Getafe CF": "Getafe",
    "Girona FC": "Girona",
    "Grêmio": "Gremio",
    "Hellas Verona": "Verona",
    "Inter Milan": "Inter",
    "Ipswich Town": "Ipswich",
    "Köln": "FC Koln",
    "Leeds United": "Leeds",
    "Leganés": "Leganes",
    "Leicester City": "Leicester",
    "Levante UD": "Levante",
    "Luton Town": "Luton",
    "Mainz 05": "Mainz",
    "Manchester United": "Manchester Utd",
    "Moreirense FC": "Moreirense",
    "Newcastle United": "Newcastle",
    "Nottingham Forest": "Nottingham",
    "Olympique Lyonnais": "Lyon",
    "Olympique Marseille": "Marseille",
    "Paris": "Paris FC",
    "Porto": "FC Porto",
    "RCD Espanyol": "Espanyol",
    "RCD Mallorca": "Mallorca",
    "Real Betis": "Betis",
    "Real Oviedo": "R. Oviedo",
    "Real Valladolid": "Valladolid",
    "Rio Ave FC": "Rio Ave",
    "Saint-Étienne": "St Etienne",
    "Sevilla FC": "Sevilla",
    "Sheffield United": "Sheffield Utd",
    "Sporting Braga": "Braga",
    "Sporting CP": "Sporting CP",
    "São Paulo": "Sao Paulo",
    "Tottenham Hotspur": "Tottenham",
    "UD Las Palmas": "Las Palmas",
    "Valencia CF": "Valencia",
    "Vasco da Gama": "Vasco",
    "Vitória": "Vitoria",
    "Vitória Guimarães": "Vitoria Guimaraes",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
}

def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def map_team_name(name):
    value = normalize_text(name)
    return FOOTYSTATS_TEAM_MAP.get(value, value)


def is_known_team_name(name):
    value = normalize_text(name)
    return value in FOOTYSTATS_TEAM_MAP or value in FOOTYSTATS_TEAM_MAP.values()

# Normalizar também a base footystats para que os nomes fiquem iguais nos dois lados do merge
dados_footystats['Home'] = dados_footystats['Home'].apply(map_team_name)
dados_footystats['Away'] = dados_footystats['Away'].apply(map_team_name)
dados_footystats['Date'] = pd.to_datetime(dados_footystats['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
datatest['Date'] = pd.to_datetime(datatest['Date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Aplicar a função de mapeamento às colunas 'Home' e 'Away' footystats
datatest['Home'] = datatest['Home'].apply(map_team_name)
datatest['Away'] = datatest['Away'].apply(map_team_name)

# Concatenar as bases de dados usando as colunas 'Home' e 'Away' com base no datatest
datatest = pd.merge(datatest, dados_footystats, on=['Date', 'League', 'Home', 'Away'], how='left')

# Eliminar linhas com valores NaN na coluna 'xG_H' após o merge
datatest = datatest.dropna(subset=['xG_H'])

# Eliminar coluna 'Goals_H_HT_y' que foi criada durante o merge e renomear a coluna 'Goals_H_HT_x' para 'Goals_H_HT'
datatest = datatest.drop(columns=['Goals_H_HT_y'])
datatest.rename(columns={'Goals_H_HT_x': 'Goals_H_HT'}, inplace=True)

# Verificar se o placar FT foi 0x1
datatest['WCS'] = datatest.apply(lambda row: 0 if row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 1 else 1, axis=1)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_CS_0x1_Lay'] - 1) if row['WCS'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)



datatest.head()

,Date,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Goals_Min_H,Odd_H_Back,Odd_A_Back,Odd_Over25_FT_Back,Odd_BTTS_Yes_Back,Odd_CS_0x1_Lay,xG_H,WCS,Profit
11,2026-04-06,ITALY 1,Udinese,Como,0,0,0.0,0.0,[],6.20,1.69,2.14,2.10,7.4,1.48,1,0.94
12,2024-09-22,ITALY 1,Fiorentina,Lazio,0,1,2.0,1.0,"[49, 90]",2.40,3.40,2.10,1.81,12.0,1.51,1,0.94
13,2024-09-15,ITALY 1,Genoa,AS Roma,0,1,1.0,1.0,[90],3.55,2.32,2.26,1.96,8.6,1.85,1,0.94
14,2024-05-05,ITALY 1,Cagliari,Lecce,1,0,1.0,1.0,[26],2.26,3.90,2.36,2.00,12.5,0.96,1,0.94
15,2024-05-12,ITALY 1,Lazio,Empoli,1,0,2.0,0.0,"[45, 89]",1.67,6.20,2.02,2.04,18.5,1.17,1,0.94


In [19]:
condicoes = [
    # Linha 1
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'] >= 20.0)),
    
    # Linha 2
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(13.0, 13.9))),
    
    # Linha 3
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(12.0, 12.9))),
    
    # Linha 4
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(18.0, 19.9))),
    
    # Linha 5
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 6
    ((datatest['Odd_H_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_A_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 7
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 8
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(14.0, 14.9))),
    
    # Linha 9
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 10
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(16.0, 17.9))),
    
    # Linha 11 xG_H > 1.10
    (datatest['xG_H'] > 1.10)
]

# Aplicar as condições: Bet = 1 se qualquer uma das condições for verdadeira
datatest['Bet'] = np.where(pd.concat(condicoes, axis=1).any(axis=1), 1, 0)

# Calcular o lucro para cada jogo com base nas condições
datatest['PL'] = np.where(
    (datatest['Bet'] == 1) & (datatest['WCS'] == 1),
    datatest['Profit'],
    np.where(
        (datatest['Bet'] == 1) & (datatest['WCS'] == 0),
        - datatest['Odd_CS_0x1_Lay'] + 1,
        0
    )
)

# Filtar apenas os jogos onde Bet = 1
datatest_bets = datatest[datatest['Bet'] == 1]

# Quantidade de apostas
total_bets = datatest_bets.shape[0]
print(f"📈 Total de Apostas: {total_bets}")

# Quantidade de apostas vencedoras
winning_bets = datatest_bets[datatest_bets['WCS'] == 1].shape[0]
print(f"✅ Apostas Vencedoras: {winning_bets}")

# Quantidade de apostas perdedoras
losing_bets = datatest_bets[datatest_bets['WCS'] == 0].shape[0]
print(f"❌ Apostas Perdedoras: {losing_bets}")
    

📈 Total de Apostas: 3591
✅ Apostas Vencedoras: 3373
❌ Apostas Perdedoras: 218


In [21]:
datatest_bets.head(20)

,Date,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Goals_Min_H,Odd_H_Back,Odd_A_Back,Odd_Over25_FT_Back,Odd_BTTS_Yes_Back,Odd_CS_0x1_Lay,xG_H,WCS,Profit,Bet,PL
11,2026-04-06,ITALY 1,Udinese,Como,0,0,0.0,0.0,[],6.20,1.69,2.14,2.10,7.4,1.48,1,0.94,1,0.94
12,2024-09-22,ITALY 1,Fiorentina,Lazio,0,1,2.0,1.0,"[49, 90]",2.40,3.40,2.10,1.81,12.0,1.51,1,0.94,1,0.94
13,2024-09-15,ITALY 1,Genoa,AS Roma,0,1,1.0,1.0,[90],3.55,2.32,2.26,1.96,8.6,1.85,1,0.94,1,0.94
14,2024-05-05,ITALY 1,Cagliari,Lecce,1,0,1.0,1.0,[26],2.26,3.90,2.36,2.00,12.5,0.96,1,0.94,1,0.94
15,2024-05-12,ITALY 1,Lazio,Empoli,1,0,2.0,0.0,"[45, 89]",1.67,6.20,2.02,2.04,18.5,1.17,1,0.94,1,0.94
16,2024-10-06,ITALY 1,Juventus,Cagliari,1,0,1.0,1.0,[15],1.44,9.60,2.04,2.36,26.0,2.06,1,0.94,1,0.94
18,2025-04-27,ITALY 1,Venezia,AC Milan,0,1,0.0,2.0,[],5.30,1.74,1.79,1.79,9.8,1.51,1,0.94,1,0.94
19,2025-04-27,ITALY 1,Como,Genoa,0,0,1.0,0.0,[59],1.91,5.00,2.30,2.08,14.5,1.48,1,0.94,1,0.94
22,2025-09-14,ITALY 1,AS Roma,Torino,0,0,0.0,1.0,[],1.56,7.80,2.08,2.22,22.0,2.27,0,-21.00,1,-21.00
23,2024-04-28,ITALY 1,Inter,Torino,0,0,2.0,0.0,"[56, 60]",1.54,8.00,2.06,2.20,22.0,1.79,1,0.94,1,0.94
